# Task_1

In [9]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
    WebDriverException,
    InvalidSessionIdException,
)
from webdriver_manager.chrome import ChromeDriverManager

print("All libraries imported successfully.")


All libraries imported successfully.


## CONFIGURATION

In [10]:
SEARCH_KEYWORD = "برنامه نویس پایتون"   # or: "دانشمند داده"

BASE_SEARCH_URL = "https://jobinja.ir/jobs?filters%5Bkeywords%5D%5B0%5D="

MIN_PAGES_TO_SCRAPE = 3

HEADLESS = False

REQUEST_DELAY = 3.0          
PAGE_LOAD_TIMEOUT = 45       
WAIT_TIMEOUT = 25            
MAX_RETRIES = 2              # retries for driver.get() before rebuilding the driver

OUTPUT_CSV_PATH = "job_listings_output.csv"

FA_DIGITS = {"1": "۱", "2": "۲", "3": "۳", "4": "۴", "5": "۵",
             "6": "۶", "7": "۷", "8": "۸", "9": "۹", "10": "۱۰"}

print(f"Search keyword: {SEARCH_KEYWORD}")
print(f"Pages to scrape: {MIN_PAGES_TO_SCRAPE}")


Search keyword: برنامه نویس پایتون
Pages to scrape: 3


## Selenium Driver Setup

In [11]:
def build_driver(headless: bool = True) -> webdriver.Chrome:
    """Create and return a freshly configured Chrome WebDriver instance."""
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--lang=fa-IR")
    # Reduces odds of the renderer process crashing on some Windows setups
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-background-networking")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    )

    service = ChromeService(ChromeDriverManager().install())
    new_driver = webdriver.Chrome(service=service, options=chrome_options)
    new_driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return new_driver


def restart_driver(old_driver):
    """Safely quit a (possibly crashed) driver and return a brand-new one."""
    try:
        old_driver.quit()
    except Exception:
        pass  # the old session may already be dead; ignore quit errors
    time.sleep(1)
    fresh_driver = build_driver(headless=HEADLESS)
    print("Chrome WebDriver was restarted with a fresh session.")
    return fresh_driver


def safe_get(driver, url: str, max_retries: int = MAX_RETRIES):
    """Navigate to `url`, retrying on timeout, and rebuilding the driver
    if the browser session itself has crashed.
    Returns the (possibly new) driver instance - always reassign the
    result back to your `driver` variable after calling this function.
    """
    for attempt in range(1, max_retries + 1):
        try:
            driver.get(url)
            return driver
        except (TimeoutException, WebDriverException, InvalidSessionIdException) as e:
            print(f"  [safe_get] Attempt {attempt}/{max_retries} failed for {url}: {type(e).__name__}")
            driver = restart_driver(driver)
    # Final attempt after the last rebuild; let the exception propagate if it fails again
    driver.get(url)
    return driver


driver = build_driver(headless=HEADLESS)
print("Chrome WebDriver launched successfully.")


Chrome WebDriver launched successfully.


## Launch

In [ ]:
def open_search_page_via_form(driver):
    driver = safe_get(driver, "https://jobinja.ir/")
    wait = WebDriverWait(driver, WAIT_TIMEOUT)

    search_box = wait.until(
        EC.presence_of_element_located((
            By.XPATH,
            '//input[contains(@placeholder, "عنوان شغلی، نام شرکت، مهارت")]'
        ))
    )
    search_box.clear()
    search_box.send_keys(SEARCH_KEYWORD)
    time.sleep(1)
    search_box.submit()

    wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "li.c-jobListView__item"))
    )
    time.sleep(REQUEST_DELAY)
    return driver


def open_search_page_via_url(driver):
    from urllib.parse import quote
    url = BASE_SEARCH_URL + quote(SEARCH_KEYWORD)
    driver = safe_get(driver, url)
    WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "li.c-jobListView__item"))
    )
    time.sleep(REQUEST_DELAY)
    return driver


try:
    driver = open_search_page_via_form(driver)
    print(f"Search page opened successfully via homepage form for: '{SEARCH_KEYWORD}'")
except (TimeoutException, NoSuchElementException, WebDriverException, InvalidSessionIdException) as e:
    print(f"Homepage search form flow failed ({type(e).__name__}). Rebuilding driver and falling back to direct search URL...")
    driver = restart_driver(driver)
    driver = open_search_page_via_url(driver)
    print(f"Search page opened successfully via direct URL for: '{SEARCH_KEYWORD}'")

print("Current URL:", driver.current_url)


Search page opened successfully via homepage form for: 'برنامه نویس پایتون'
Current URL: https://jobinja.ir/jobs?filters%5Bkeywords%5D%5B%5D=%D8%A8%D8%B1%D9%86%D8%A7%D9%85%D9%87+%D9%86%D9%88%DB%8C%D8%B3+%D9%BE%D8%A7%DB%8C%D8%AA%D9%88%D9%86&filters%5Blocations%5D%5B%5D=&filters%5Bjob_categories%5D%5B%5D=&b=


## Extraction

In [ ]:
def to_fa_digits(n: int) -> str:
    return FA_DIGITS.get(str(n), str(n))


def extract_jobs_from_current_page(driver) -> list:
    """Parse the current page's HTML with BeautifulSoup and extract
    job title, company name, and location for every job card.
    """
    soup = BeautifulSoup(driver.page_source, "html.parser")
    jobs_on_page = []

    job_cards = soup.select("li.c-jobListView__item")

    for card in job_cards:
        title_tag = card.select_one("h2.c-jobListView__title a.c-jobListView__titleLink")
        job_title = title_tag.get_text(strip=True) if title_tag else None
        if not job_title:
            continue

        meta_items = card.select("ul.c-jobListView__meta li.c-jobListView__metaItem")
        company_name = meta_items[0].get_text(strip=True) if len(meta_items) > 0 else None
        location = meta_items[1].get_text(strip=True) if len(meta_items) > 1 else None

        jobs_on_page.append({
            "job_title": job_title,
            "company_name": company_name,
            "location": location,
        })

    return jobs_on_page


def go_to_next_page(driver, current_page: int) -> bool:
    """Click the numbered pagination link for (current_page + 1)."""
    next_page_num = str(current_page + 1)
    next_page_fa = to_fa_digits(current_page + 1)

    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.5)
        next_page_button = WebDriverWait(driver, WAIT_TIMEOUT).until(
            EC.presence_of_element_located((
                By.XPATH,
                f'//a[contains(@href, "page={next_page_num}") '
                f'or normalize-space(text())="{next_page_num}" '
                f'or normalize-space(text())="{next_page_fa}"]'
            ))
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_page_button)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_page_button)
        time.sleep(REQUEST_DELAY)
        return True
    except (TimeoutException, NoSuchElementException, ElementClickInterceptedException):
        return False
    except (WebDriverException, InvalidSessionIdException):
        # The browser crashed in the middle of pagination 
        # the callback loop detects this problem via the outer try/except function and stops instead of incrementing.
        raise


all_jobs = []
current_page = 1

while current_page <= MIN_PAGES_TO_SCRAPE:
    print(f"Scraping page {current_page}...")
    try:
        WebDriverWait(driver, WAIT_TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "li.c-jobListView__item"))
        )
    except TimeoutException:
        print("No job cards found on this page. Stopping.")
        break

    page_jobs = extract_jobs_from_current_page(driver)
    print(f"  -> Found {len(page_jobs)} job listings on page {current_page}.")
    all_jobs.extend(page_jobs)

    if current_page < MIN_PAGES_TO_SCRAPE:
        try:
            moved = go_to_next_page(driver, current_page)
        except (WebDriverException, InvalidSessionIdException):
            print("Browser session crashed while paginating. Stopping with data collected so far.")
            break
        if not moved:
            print("Could not navigate to the next page. Reached the last available page.")
            break

    current_page += 1

print(f"\nTotal jobs scraped: {len(all_jobs)}")


Scraping page 1...
  -> Found 20 job listings on page 1.
Scraping page 2...
  -> Found 20 job listings on page 2.
Scraping page 3...
  -> Found 20 job listings on page 3.

Total jobs scraped: 60


### Close the browser

In [ ]:
try:
    driver.quit()
    print("Browser closed.")
except Exception:
    print("Browser was already closed or crashed; nothing to do.")


Browser closed.


## Output

### Save results to a pandas DataFrame

In [ ]:
df_jobs = pd.DataFrame(all_jobs)

for col in df_jobs.columns:
    df_jobs[col] = df_jobs[col].astype(str).str.strip()

df_jobs.drop_duplicates(inplace=True)
df_jobs.reset_index(drop=True, inplace=True)

print(f"Final DataFrame shape: {df_jobs.shape}")
df_jobs.head(15)


Final DataFrame shape: (60, 3)


,job_title,company_name,location
0,توسعه دهنده پایتون (Python Software Engineer-A...,رهند هوشمند نوین داده | Rahand Hoshmand Novin ...,تهران، تهران
1,برنامه‌نویس پایتون (Python/Django-اصفهان),دانش بنیان مهندسی ارتباطی پیام پرداز | Payam P...,اصفهان، اصفهان
2,برنامه‌نویس Python,همیان | Hamyan,تهران، تهران
3,طراح صنعتی(فلاسک-مشهد-دورکاری),استودیو کوروش | Cyrus Studio,خراسان رضوی، مشهد
4,برنامه‌نویس ارشد پایتون,نت‌بان شریف | NetBaan Sharif,تهران، تهران
5,توسعه‌دهنده بک‌اند (Python-Django-کرج),میزگرد | Mizegerd,البرز، کرج
6,Python Developer,آرتا رای انرژی | Arta Ray Energy,تهران، تهران
7,Python Developer,گروه مالی هامرز | Hummers Group,تهران، تهران
8,برنامه‌نویس (Python / Django),فناوری اطلاعات رسپینا رادین | Respina Radyn IT Co,تهران، تهران
9,برنامه‌نویس ارشد پایتون (دورکاری),پایا | RCP,تهران، تهران


### Export to CSV

In [ ]:
df_jobs.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Data exported successfully to: {OUTPUT_CSV_PATH}")


Data exported successfully to: job_listings_output.csv
